# 실습 3. NSL-KDD 이상탐지 → 임베딩 RAG → Qwen HTML

## 사용 데이터
`Exercise3. nsl_kdd_sample.csv`

## 전체 흐름

```text
CSV 업로드
→ 범주형 One-Hot Encoding
→ 수치형 표준화
→ Isolation Forest 이상탐지
→ 이상 점수 상위 5건 CSV 저장
→ 임베딩 RAG 검색
→ Qwen 분석 보고서
→ Qwen HTML 대시보드 생성
```

In [1]:
# ============================================================
# 1. 라이브러리 설치
# ============================================================
!pip -q install -U sentence-transformers transformers accelerate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.7/596.7 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 24.2 MB/s eta 0:00:00


In [2]:
# ============================================================
# 2. 라이브러리 불러오기 및 실행 장치 확인
# ============================================================

import re
import json
import html
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import torch

from google.colab import files
from IPython.display import display, HTML

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import confusion_matrix, classification_report

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("라이브러리 로드 완료")
print("현재 실행 장치:", DEVICE)



라이브러리 로드 완료
현재 실행 장치: cpu


In [3]:
# ============================================================
# 3. Exercise3 CSV 업로드
# ============================================================

print("Exercise3. nsl_kdd_sample.csv 파일을 업로드하세요.")

uploaded = files.upload()

csv_files = [
    name for name in uploaded.keys()
    if name.lower().endswith(".csv")
]

if not csv_files:
    raise ValueError("CSV 파일이 업로드되지 않았습니다.")

file_name = csv_files[0]
df = pd.read_csv(file_name)

print("업로드 파일:", file_name)
print("데이터 크기:", df.shape)
display(df.head())

Exercise3. nsl_kdd_sample.csv 파일을 업로드하세요.


Saving Exercise3. nsl_kdd_sample.csv to Exercise3. nsl_kdd_sample.csv
업로드 파일: Exercise3. nsl_kdd_sample.csv
데이터 크기: (3000, 44)


,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,root_shell,su_attempted,num_root,num_file_creations,num_shells,num_access_files,num_outbound_cmds,is_host_login,is_guest_login,count,srv_count,serror_rate,srv_serror_rate,rerror_rate,srv_rerror_rate,same_srv_rate,diff_srv_rate,srv_diff_host_rate,dst_host_count,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,label,difficulty,binary_label
0,0,tcp,bgp,S0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,217,2,1.0,1.0,0.0,0.0,0.01,0.07,0.0,255,2,0.01,0.08,0.00,0.00,1.0,1.0,0.0,0.00,neptune,19,attack
1,0,tcp,ftp_data,SF,201,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,1,0.0,0.0,0.0,0.0,1.00,0.00,0.0,94,70,0.19,0.76,0.18,0.03,0.0,0.0,0.7,0.01,normal,21,normal
2,0,tcp,private,S0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,119,14,1.0,1.0,0.0,0.0,0.12,0.08,0.0,255,20,0.08,0.07,0.00,0.00,1.0,1.0,0.0,0.00,neptune,21,attack
3,0,tcp,http,S1,235,21900,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,2,1.0,0.5,0.0,0.0,1.00,0.00,1.0,255,255,1.00,0.00,0.00,0.00,0.0,0.0,0.0,0.00,normal,20,normal
4,0,tcp,private,S0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,289,5,1.0,1.0,0.0,0.0,0.02,0.06,0.0,255,7,0.03,0.08,0.00,0.00,1.0,1.0,0.0,0.00,neptune,21,attack


In [4]:
# ============================================================
# 4. NSL-KDD 데이터 구조 검증
# ============================================================

required_columns = [
    "duration",
    "protocol_type",
    "service",
    "flag",
    "src_bytes",
    "dst_bytes",
    "count",
    "srv_count",
    "label",
    "difficulty",
    "binary_label"
]

missing_columns = [
    column for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        "Exercise3 NSL-KDD CSV가 아닙니다.\n"
        f"누락된 컬럼: {missing_columns}\n"
        f"현재 컬럼: {df.columns.tolist()}"
    )

print("Exercise3 CSV 구조 확인 완료")
print("전체 컬럼 수:", len(df.columns))

display(df["binary_label"].value_counts())

Exercise3 CSV 구조 확인 완료
전체 컬럼 수: 44


,count
binary_label,
attack,1500
normal,1500


In [5]:
# ============================================================
# 5. 이상탐지 입력 데이터 전처리
# ============================================================
# label, binary_label, difficulty는 정답 또는 부가정보이므로
# Isolation Forest 학습 입력에서는 제외합니다.
# ============================================================

exclude_columns = [
    "label",
    "binary_label",
    "difficulty"
]

X = df.drop(columns=exclude_columns).copy()

X = X.replace([np.inf, -np.inf], np.nan)

numeric_columns = X.select_dtypes(include=np.number).columns
categorical_columns = X.select_dtypes(exclude=np.number).columns

for column in numeric_columns:
    median_value = X[column].median()

    if pd.isna(median_value):
        median_value = 0

    X[column] = X[column].fillna(median_value)

for column in categorical_columns:
    X[column] = X[column].fillna("Unknown").astype(str)

X_encoded = pd.get_dummies(
    X,
    columns=categorical_columns,
    dtype=int
)

X_encoded = X_encoded.apply(
    pd.to_numeric,
    errors="coerce"
).fillna(0)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_encoded)

print("원본 Feature 수:", X.shape[1])
print("인코딩 후 Feature 수:", X_encoded.shape[1])
print("모델 입력 크기:", X_scaled.shape)

원본 Feature 수: 41
인코딩 후 Feature 수: 109
모델 입력 크기: (3000, 109)


In [6]:
# ============================================================
# 6. Isolation Forest 학습 및 예측
# ============================================================

isolation_model = IsolationForest(
    n_estimators=200,
    contamination=0.05,
    random_state=42,
    n_jobs=-1
)

isolation_model.fit(X_scaled)

raw_prediction = isolation_model.predict(X_scaled)
anomaly_score = isolation_model.decision_function(X_scaled)

result_df = df.copy()
result_df["model_prediction"] = np.where(
    raw_prediction == -1,
    "anomaly",
    "normal"
)
result_df["anomaly_score"] = anomaly_score

display(result_df["model_prediction"].value_counts())

,count
model_prediction,
normal,2850
anomaly,150


In [7]:
# ============================================================
# 7. 실제 binary_label과 참고 비교
# ============================================================
# 이 평가는 모델 학습에 binary_label을 사용한 것이 아니라
# 비지도학습 완료 후 결과를 비교하는 참고 단계입니다.
# ============================================================

actual = (
    result_df["binary_label"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map(
        {
            "normal": 0,
            "attack": 1
        }
    )
)

predicted = np.where(
    result_df["model_prediction"] == "anomaly",
    1,
    0
)

valid = actual.notna()

print("[혼동행렬]")
print(confusion_matrix(actual[valid], predicted[valid]))

print("\n[분류 보고서]")
print(
    classification_report(
        actual[valid],
        predicted[valid],
        target_names=["정상", "공격"],
        zero_division=0
    )
)

[혼동행렬]
[[1476   24]
 [1374  126]]

[분류 보고서]
              precision    recall  f1-score   support

          정상       0.52      0.98      0.68      1500
          공격       0.84      0.08      0.15      1500

    accuracy                           0.53      3000
   macro avg       0.68      0.53      0.42      3000
weighted avg       0.68      0.53      0.42      3000



In [8]:
# ============================================================
# 8. 가장 이상한 이벤트 5건 추출 및 CSV 저장
# ============================================================

top5_df = (
    result_df
    .sort_values("anomaly_score", ascending=True)
    .head(5)
    .copy()
    .reset_index(drop=True)
)

top5_df.insert(
    0,
    "event_id",
    [f"NSL-ANOM-{i:03d}" for i in range(1, 6)]
)

top5_df["detected_at"] = datetime.now().strftime(
    "%Y-%m-%d %H:%M:%S"
)

top5_df.to_csv(
    "exercise3_nsl_kdd_top5_anomalies.csv",
    index=False,
    encoding="utf-8-sig"
)

display(
    top5_df[
        [
            "event_id",
            "protocol_type",
            "service",
            "flag",
            "src_bytes",
            "dst_bytes",
            "count",
            "srv_count",
            "anomaly_score",
            "label",
            "binary_label"
        ]
    ]
)

,event_id,protocol_type,service,flag,src_bytes,dst_bytes,count,srv_count,anomaly_score,label,binary_label
0,NSL-ANOM-001,tcp,telnet,SF,4113,51633,1,1,-0.087978,ipsweep,attack
1,NSL-ANOM-002,tcp,telnet,SF,3555,198172,1,1,-0.071049,normal,normal
2,NSL-ANOM-003,tcp,ftp,RSTO,0,61,1,1,-0.064103,ipsweep,attack
3,NSL-ANOM-004,tcp,telnet,SF,794,152110,1,1,-0.054494,normal,normal
4,NSL-ANOM-005,tcp,telnet,SF,1412,25260,1,1,-0.042349,multihop,attack


## 센터형 RAG 지식베이스

기존처럼 짧은 대응문서 몇 개만 사용하는 대신, 센터에서 지속적으로 축적·관리할 수 있는 지식항목으로 구성합니다.

각 지식문서는 다음 필드를 포함합니다.

- `doc_type`: 침해사고 사례, 대응절차, 분석 체크리스트, 보고기준
- `attack_category`: 네트워크 공격 유형
- `summary`: 사고 또는 절차 개요
- `detection_points`: 탐지·판단 근거
- `response_steps`: 초동조치 및 대응절차
- `evidence_to_collect`: 확보해야 할 증적
- `escalation_criteria`: 보고 및 상급부대 격상 기준
- `lessons_learned`: 유사 사고 교훈
- `owner`: 담당 기능
- `updated_at`: 최종 갱신일

노트북 실행 시 기본 지식베이스가 CSV로 저장되므로, 이후 센터 내부 승인자료로 내용을 추가·수정하여 재사용할 수 있습니다.

In [9]:
# ============================================================
# 9. 센터형 네트워크 침해사고 RAG 지식베이스
# ============================================================
# 일반화된 교육용 사례와 절차입니다.
# 실제 운용 시에는 승인된 내부 침해사고 보고서와 SOP를
# 동일한 컬럼 구조로 추가하여 유지·관리할 수 있습니다.
# ============================================================

knowledge_documents = [
    {
        "doc_id": "NET-IR-001",
        "doc_type": "침해사고 사례",
        "title": "대량 SYN 연결을 이용한 서비스 거부 사고",
        "attack_category": "DoS / SYN Flood",
        "summary": (
            "특정 서비스로 단시간에 대량의 SYN 연결이 유입되어 "
            "연결 대기열과 서비스 응답시간이 증가한 사례이다."
        ),
        "detection_points": (
            "count·srv_count 증가, serror_rate·srv_serror_rate 상승, "
            "동일 목적지 서비스 집중, 정상 세션 완료율 저하를 확인한다."
        ),
        "response_steps": (
            "대상 서비스 가용성 확인, 출발지 분포 분석, 임계치 기반 차단, "
            "SYN 보호 설정, 상위 구간 협조 및 지속 모니터링을 수행한다."
        ),
        "evidence_to_collect": (
            "방화벽·IPS·서버 연결 로그, NetFlow, 패킷 캡처, 서비스 상태, "
            "출발지·목적지·포트·시간대별 통계를 수집한다."
        ),
        "escalation_criteria": (
            "중요 서비스 중단, 다수 출발지 분산공격, 작전 영향 또는 "
            "장시간 지속 시 긴급 보고 및 상급 협조를 요청한다."
        ),
        "lessons_learned": (
            "단일 출발지 차단만으로 충분하지 않으며 서비스 보호정책과 "
            "상위망 협조 절차를 함께 준비해야 한다."
        ),
        "owner": "보안관제/네트워크운영",
        "updated_at": "2026-08-01",
        "keywords": "SYN Flood neptune serror count srv_count DoS"
    },
    {
        "doc_id": "NET-IR-002",
        "doc_type": "침해사고 사례",
        "title": "ICMP 기반 대량 트래픽 유입 사고",
        "attack_category": "DoS / ICMP Flood",
        "summary": (
            "대량 ICMP 트래픽으로 회선과 장비 자원이 소모된 사례이다."
        ),
        "detection_points": (
            "ICMP 프로토콜 집중, 짧은 연결 지속시간, 높은 패킷 수, "
            "다수 목적지 또는 특정 목적지 집중을 확인한다."
        ),
        "response_steps": (
            "ICMP 사용 필요성 확인, 비정상 유형·크기 제한, 장비 부하 확인, "
            "출발지 및 상위망 차단 협조를 수행한다."
        ),
        "evidence_to_collect": (
            "인터페이스 사용률, NetFlow, 패킷 크기·유형·코드, "
            "장비 CPU·메모리와 드롭 통계를 수집한다."
        ),
        "escalation_criteria": (
            "회선 포화, 중요 구간 장애, 다수 장비 영향 시 즉시 격상한다."
        ),
        "lessons_learned": (
            "정상 진단용 ICMP와 공격 트래픽을 구분할 수 있도록 "
            "유형·속도 기반 기준을 마련해야 한다."
        ),
        "owner": "네트워크운영/보안관제",
        "updated_at": "2026-08-01",
        "keywords": "ICMP smurf flood bandwidth"
    },
    {
        "doc_id": "NET-IR-003",
        "doc_type": "침해사고 사례",
        "title": "다수 포트 스캔 후 원격서비스 접근 시도",
        "attack_category": "Probe / Port Scan",
        "summary": (
            "외부 호스트가 여러 포트를 탐색한 뒤 SSH·FTP 등 "
            "원격서비스로 접근을 시도한 사례이다."
        ),
        "detection_points": (
            "diff_srv_rate 증가, 여러 서비스 접근, 짧은 세션 반복, "
            "동일 출발지의 다수 목적지 또는 포트 접근을 확인한다."
        ),
        "response_steps": (
            "출발지 차단, 노출 서비스 확인, 불필요 포트 폐쇄, "
            "후속 인증시도 및 취약점 공격 로그를 연계 분석한다."
        ),
        "evidence_to_collect": (
            "방화벽 허용·차단 로그, 포트 목록, 대상 호스트, "
            "IDS 경보, 인증 로그와 시간대별 접근 순서를 수집한다."
        ),
        "escalation_criteria": (
            "스캔 후 로그인 성공, 취약점 악용, 내부 중요자산 접근 시 격상한다."
        ),
        "lessons_learned": (
            "스캔 단독 이벤트보다 후속 인증·침투 행위와 연결하여 판단해야 한다."
        ),
        "owner": "보안관제",
        "updated_at": "2026-08-01",
        "keywords": "portsweep ipsweep nmap probe diff_srv_rate"
    },
    {
        "doc_id": "NET-IR-004",
        "doc_type": "침해사고 사례",
        "title": "FTP 계정 무차별 대입 및 로그인 성공",
        "attack_category": "R2L / Brute Force",
        "summary": (
            "FTP 서비스에 반복 인증 실패가 발생한 뒤 특정 계정으로 "
            "로그인에 성공하고 파일 접근이 이어진 사례이다."
        ),
        "detection_points": (
            "num_failed_logins 증가, 로그인 성공 전 반복 실패, 동일 출발지 집중, "
            "성공 이후 src_bytes·dst_bytes 변화와 파일 전송을 확인한다."
        ),
        "response_steps": (
            "계정 잠금, 비밀번호 초기화, 출발지 차단, FTP 세션 종료, "
            "접근 파일과 후속 내부 이동 여부를 조사한다."
        ),
        "evidence_to_collect": (
            "FTP 인증·전송 로그, 계정 변경 이력, 파일 목록·해시, "
            "방화벽 및 호스트 로그를 수집한다."
        ),
        "escalation_criteria": (
            "로그인 성공, 중요파일 접근·유출, 공용 또는 권한계정 사용 시 격상한다."
        ),
        "lessons_learned": (
            "실패 횟수뿐 아니라 성공 전환과 이후 행위를 함께 탐지해야 한다."
        ),
        "owner": "계정관리/침해대응",
        "updated_at": "2026-08-01",
        "keywords": "guess_passwd ftp failed_logins login file transfer"
    },
    {
        "doc_id": "NET-IR-005",
        "doc_type": "침해사고 사례",
        "title": "취약 서비스 악용 후 원격 셸 획득",
        "attack_category": "R2L / Exploitation",
        "summary": (
            "외부에서 취약한 네트워크 서비스를 악용하여 원격 명령 실행 및 "
            "셸 접근을 획득한 사례이다."
        ),
        "detection_points": (
            "비정상 서비스 요청, 오류 응답 뒤 세션 변화, 로그인 없이 명령 실행, "
            "서비스 계정의 프로세스 생성 여부를 확인한다."
        ),
        "response_steps": (
            "대상 호스트 격리, 취약 서비스 중지, 프로세스·네트워크 연결 보존, "
            "메모리·디스크 증적 확보 및 동일 취약점 노출 자산을 점검한다."
        ),
        "evidence_to_collect": (
            "서비스 로그, EDR·프로세스 트리, 명령 이력, 메모리, 파일 해시, "
            "패킷 및 외부 연결정보를 수집한다."
        ),
        "escalation_criteria": (
            "원격명령 실행, 지속성 확보, 중요자산 침해 또는 내부 확산 시 긴급 격상한다."
        ),
        "lessons_learned": (
            "네트워크 경보와 호스트 행위를 연계해야 실제 침투 여부를 확인할 수 있다."
        ),
        "owner": "침해대응/시스템운영",
        "updated_at": "2026-08-01",
        "keywords": "remote shell exploit R2L service command"
    },
    {
        "doc_id": "NET-IR-006",
        "doc_type": "침해사고 사례",
        "title": "버퍼오버플로우를 이용한 권한상승 사고",
        "attack_category": "U2R / Privilege Escalation",
        "summary": (
            "일반 권한 접근 이후 취약 프로그램을 이용해 관리자 권한을 획득한 사례이다."
        ),
        "detection_points": (
            "root_shell, num_root, su_attempted, 비정상 프로세스 권한, "
            "충돌 이후 권한 변경과 시스템 파일 수정을 확인한다."
        ),
        "response_steps": (
            "호스트 격리, 권한계정 보호, 프로세스·메모리 보존, "
            "취약 바이너리와 변경 파일 분석, 재설치 또는 복구를 검토한다."
        ),
        "evidence_to_collect": (
            "프로세스 권한, 감사로그, 메모리 덤프, 코어덤프, "
            "바이너리 해시, 시스템 파일 변경 이력을 수집한다."
        ),
        "escalation_criteria": (
            "관리자 권한 획득, 보안기능 무력화, 중요정보 접근 시 긴급 격상한다."
        ),
        "lessons_learned": (
            "권한상승은 네트워크 특징만으로 확인하기 어려우므로 "
            "호스트 감사로그와 메모리 분석이 필요하다."
        ),
        "owner": "침해대응/포렌식",
        "updated_at": "2026-08-01",
        "keywords": "buffer_overflow root_shell num_root U2R"
    },
    {
        "doc_id": "NET-IR-007",
        "doc_type": "침해사고 사례",
        "title": "루트킷 설치 및 보안로그 은폐",
        "attack_category": "U2R / Rootkit",
        "summary": (
            "권한상승 이후 루트킷을 설치하여 프로세스와 파일을 숨기고 "
            "로그를 삭제한 사례이다."
        ),
        "detection_points": (
            "시스템 명령 결과와 원시 디스크 정보 불일치, 로그 공백, "
            "커널 모듈 변화, 비정상 권한 파일을 확인한다."
        ),
        "response_steps": (
            "네트워크 격리, 메모리 우선 확보, 신뢰 가능한 외부매체로 점검, "
            "중요 증적 보존 후 시스템 재구축을 검토한다."
        ),
        "evidence_to_collect": (
            "메모리 이미지, 커널 모듈, 파일시스템 메타데이터, "
            "감사로그, EDR 원격 수집자료를 확보한다."
        ),
        "escalation_criteria": (
            "보안기능 은폐, 로그 삭제, 관리자 지속성 또는 중요시스템 침해 시 격상한다."
        ),
        "lessons_learned": (
            "침해된 호스트의 명령 결과를 그대로 신뢰하지 말고 외부 검증이 필요하다."
        ),
        "owner": "디지털포렌식",
        "updated_at": "2026-08-01",
        "keywords": "rootkit hidden process log deletion kernel"
    },
    {
        "doc_id": "NET-IR-008",
        "doc_type": "침해사고 사례",
        "title": "비정상 대용량 외부 전송 의심 사고",
        "attack_category": "Data Exfiltration",
        "summary": (
            "정상 업무시간 외 특정 외부 목적지로 대용량 데이터가 전송된 사례이다."
        ),
        "detection_points": (
            "src_bytes 또는 dst_bytes 급증, 평소 사용하지 않는 서비스, "
            "장시간 세션, 신규 외부 목적지와 반복 전송을 확인한다."
        ),
        "response_steps": (
            "세션 차단, 대상 호스트 격리, 전송 파일 및 사용자 확인, "
            "동일 목적지 연결 검색과 계정·단말 조사를 수행한다."
        ),
        "evidence_to_collect": (
            "NetFlow, 프록시·방화벽 로그, 파일 접근 이력, 전송 파일 해시, "
            "사용자 활동과 외부 목적지 정보를 수집한다."
        ),
        "escalation_criteria": (
            "중요자료 포함 가능성, 암호화된 대량 전송, 다수 호스트 연계 시 즉시 격상한다."
        ),
        "lessons_learned": (
            "트래픽 양뿐 아니라 사용자·자산 기준선과 시간대, 목적지 신규성을 함께 평가한다."
        ),
        "owner": "DLP/침해대응",
        "updated_at": "2026-08-01",
        "keywords": "exfiltration src_bytes dst_bytes large transfer"
    },
    {
        "doc_id": "NET-SOP-001",
        "doc_type": "대응절차",
        "title": "네트워크 이상 이벤트 초동조치 절차",
        "attack_category": "Common",
        "summary": (
            "머신러닝 또는 보안장비에서 탐지된 이상 이벤트를 확인하고 "
            "사건으로 전환하기 위한 공통 초동절차이다."
        ),
        "detection_points": (
            "탐지시각, 출발지·목적지, 프로토콜·서비스, 이상 점수, "
            "자산 중요도, 동일 이벤트 반복과 다른 보안경보 연계를 확인한다."
        ),
        "response_steps": (
            "원본 로그 보존 → 자산·사용자 식별 → 통신 흐름 확인 → "
            "차단 필요성 판단 → 호스트 확인 → 영향범위 조사 → 상황보고 순으로 수행한다."
        ),
        "evidence_to_collect": (
            "원본 이벤트, 방화벽·IDS·NetFlow·DNS·인증·호스트 로그, "
            "패킷 캡처와 분석자 조치기록을 확보한다."
        ),
        "escalation_criteria": (
            "중요자산 관련, 침투 성공, 서비스 장애, 정보유출, "
            "내부 확산 또는 반복 캠페인 정황 시 격상한다."
        ),
        "lessons_learned": (
            "이상 점수만으로 공격을 확정하지 말고 다중 로그와 자산 맥락을 결합한다."
        ),
        "owner": "보안관제",
        "updated_at": "2026-08-01",
        "keywords": "초동조치 이상탐지 로그보존 영향범위"
    },
    {
        "doc_id": "NET-SOP-002",
        "doc_type": "대응절차",
        "title": "네트워크 차단 및 예외 승인 절차",
        "attack_category": "Containment",
        "summary": (
            "침해 의심 통신을 차단하면서 정상 업무 영향을 최소화하기 위한 절차이다."
        ),
        "detection_points": (
            "차단 대상 IP·도메인·포트의 악성 근거, 자산 중요도, "
            "정상 업무 사용 여부와 대체경로를 확인한다."
        ),
        "response_steps": (
            "임시 차단 → 영향 모니터링 → 담당부서 확인 → 정식 정책 반영 또는 해제 → "
            "변경기록 및 사후검토를 수행한다."
        ),
        "evidence_to_collect": (
            "차단 근거, 정책 변경자·시간, 적용 장비, 영향도, "
            "예외 승인 내역과 해제 조건을 기록한다."
        ),
        "escalation_criteria": (
            "중요서비스 영향 가능성, 광범위 차단, 작전 관련 통신 포함 시 승인권자에게 격상한다."
        ),
        "lessons_learned": (
            "차단은 기술조치뿐 아니라 변경관리와 영향평가가 함께 이루어져야 한다."
        ),
        "owner": "네트워크운영/보안관제",
        "updated_at": "2026-08-01",
        "keywords": "차단 예외 승인 변경관리 방화벽"
    },
    {
        "doc_id": "NET-CHK-001",
        "doc_type": "분석 체크리스트",
        "title": "DoS 이상 이벤트 분석 체크리스트",
        "attack_category": "DoS",
        "summary": (
            "대량 연결 또는 오류율 증가 이벤트의 실제 서비스 영향과 공격 여부를 확인한다."
        ),
        "detection_points": (
            "연결 수, 오류율, 동시 세션, 출발지 분산도, 대상 서비스 부하, "
            "정상 기준선 대비 증가율을 확인한다."
        ),
        "response_steps": (
            "서비스 상태 확인, 시간대별 추세 분석, 출발지 그룹화, "
            "정상 트래픽 제외, 임계치 및 차단정책 적용을 검토한다."
        ),
        "evidence_to_collect": (
            "NetFlow, 패킷, 장비 성능, 서버 연결 상태와 애플리케이션 응답시간을 수집한다."
        ),
        "escalation_criteria": (
            "서비스 지연·중단, 회선 포화, 다수 출발지 또는 장시간 지속 시 격상한다."
        ),
        "lessons_learned": (
            "트래픽 증가만으로 공격을 단정하지 말고 실제 서비스 영향과 정상 행사 여부를 확인한다."
        ),
        "owner": "보안관제/네트워크운영",
        "updated_at": "2026-08-01",
        "keywords": "DoS checklist baseline service impact"
    },
    {
        "doc_id": "NET-CHK-002",
        "doc_type": "분석 체크리스트",
        "title": "인증 이상 이벤트 분석 체크리스트",
        "attack_category": "Authentication",
        "summary": (
            "로그인 실패 및 비정상 로그인 성공 이벤트를 조사하는 체크리스트이다."
        ),
        "detection_points": (
            "실패 횟수, 계정 수, 출발지 수, 성공 전환, 시간대, "
            "신규 위치·장비와 권한계정 여부를 확인한다."
        ),
        "response_steps": (
            "계정 잠금 필요성 판단, 사용자 확인, 비밀번호·MFA 조치, "
            "성공 로그인 이후 활동과 연관 계정을 조사한다."
        ),
        "evidence_to_collect": (
            "인증 로그, 계정 변경, MFA 이벤트, 세션, 파일·메일 접근, "
            "VPN 및 원격접속 로그를 수집한다."
        ),
        "escalation_criteria": (
            "권한계정 로그인 성공, 중요자산 접근, 다수 계정 공격 시 격상한다."
        ),
        "lessons_learned": (
            "실패 이벤트와 성공 이후 행위를 하나의 타임라인으로 연결해야 한다."
        ),
        "owner": "계정관리/보안관제",
        "updated_at": "2026-08-01",
        "keywords": "authentication failed login MFA account"
    },
    {
        "doc_id": "NET-RPT-001",
        "doc_type": "보고기준",
        "title": "네트워크 침해사고 상황보고 기준",
        "attack_category": "Reporting",
        "summary": (
            "이상 이벤트의 영향도와 확인 수준에 따라 보고 단계를 결정하는 기준이다."
        ),
        "detection_points": (
            "공격 성공 여부, 자산 중요도, 서비스 영향, 정보유출, "
            "내부 확산, 공격 지속성과 피해 범위를 평가한다."
        ),
        "response_steps": (
            "확인 사실, 분석 근거, 추정, 미확인 사항, 조치현황, "
            "추가 계획을 구분하여 시간순으로 보고한다."
        ),
        "evidence_to_collect": (
            "타임라인, IOC, 관련 자산·계정, 로그 위치, 차단 내역과 "
            "담당부서 협조사항을 첨부한다."
        ),
        "escalation_criteria": (
            "중요 작전·업무 영향, 관리자 권한 탈취, 정보유출, "
            "다수 시스템 확산 또는 외부기관 협조 필요 시 긴급 격상한다."
        ),
        "lessons_learned": (
            "머신러닝 이상탐지 결과와 실제 침해 확인 수준을 구분하여 보고한다."
        ),
        "owner": "상황보고/침해대응",
        "updated_at": "2026-08-01",
        "keywords": "상황보고 격상 영향도 타임라인 침해"
    },
    {
        "doc_id": "NET-LL-001",
        "doc_type": "교훈",
        "title": "이상탐지 오탐 감소 및 기준선 관리 교훈",
        "attack_category": "Detection Engineering",
        "summary": (
            "정기 점검, 백업, 대규모 배포 등 정상 업무가 이상으로 탐지된 사례의 교훈이다."
        ),
        "detection_points": (
            "업무 일정, 자산 역할, 사용자, 시간대, 정상 트래픽 기준선과 "
            "변경작업 승인 여부를 확인한다."
        ),
        "response_steps": (
            "오탐 원인 기록, 허용조건 구체화, 탐지 특성 보완, "
            "예외 만료일 설정과 재검토를 수행한다."
        ),
        "evidence_to_collect": (
            "변경작업 승인, 업무 일정, 정상 기준선, 오탐 이벤트와 "
            "탐지규칙 변경 이력을 수집한다."
        ),
        "escalation_criteria": (
            "예외 범위가 넓어 탐지 공백을 만들거나 동일 오탐이 반복되면 탐지개발로 격상한다."
        ),
        "lessons_learned": (
            "예외를 무기한 적용하지 말고 자산·시간·행위 조건과 만료일을 명확히 관리한다."
        ),
        "owner": "탐지개발/보안관제",
        "updated_at": "2026-08-01",
        "keywords": "오탐 baseline 예외 변경작업 탐지규칙"
    }
]

kb_df = pd.DataFrame(knowledge_documents)

search_columns = [
    "doc_type",
    "title",
    "attack_category",
    "summary",
    "detection_points",
    "response_steps",
    "evidence_to_collect",
    "escalation_criteria",
    "lessons_learned",
    "keywords"
]

kb_df["search_text"] = (
    kb_df[search_columns]
    .fillna("")
    .astype(str)
    .agg(" ".join, axis=1)
)

kb_df.drop(columns=["search_text"]).to_csv(
    "center_network_incident_rag_knowledge.csv",
    index=False,
    encoding="utf-8-sig"
)

print("센터형 네트워크 RAG 지식베이스 생성 완료")
print("문서 수:", len(kb_df))
print("저장 파일: center_network_incident_rag_knowledge.csv")

display(
    kb_df[
        [
            "doc_id",
            "doc_type",
            "title",
            "attack_category",
            "owner",
            "updated_at"
        ]
    ]
)

센터형 네트워크 RAG 지식베이스 생성 완료
문서 수: 14
저장 파일: center_network_incident_rag_knowledge.csv


,doc_id,doc_type,title,attack_category,owner,updated_at
0,NET-IR-001,침해사고 사례,대량 SYN 연결을 이용한 서비스 거부 사고,DoS / SYN Flood,보안관제/네트워크운영,2026-08-01
1,NET-IR-002,침해사고 사례,ICMP 기반 대량 트래픽 유입 사고,DoS / ICMP Flood,네트워크운영/보안관제,2026-08-01
2,NET-IR-003,침해사고 사례,다수 포트 스캔 후 원격서비스 접근 시도,Probe / Port Scan,보안관제,2026-08-01
3,NET-IR-004,침해사고 사례,FTP 계정 무차별 대입 및 로그인 성공,R2L / Brute Force,계정관리/침해대응,2026-08-01
4,NET-IR-005,침해사고 사례,취약 서비스 악용 후 원격 셸 획득,R2L / Exploitation,침해대응/시스템운영,2026-08-01
5,NET-IR-006,침해사고 사례,버퍼오버플로우를 이용한 권한상승 사고,U2R / Privilege Escalation,침해대응/포렌식,2026-08-01
6,NET-IR-007,침해사고 사례,루트킷 설치 및 보안로그 은폐,U2R / Rootkit,디지털포렌식,2026-08-01
7,NET-IR-008,침해사고 사례,비정상 대용량 외부 전송 의심 사고,Data Exfiltration,DLP/침해대응,2026-08-01
8,NET-SOP-001,대응절차,네트워크 이상 이벤트 초동조치 절차,Common,보안관제,2026-08-01
9,NET-SOP-002,대응절차,네트워크 차단 및 예외 승인 절차,Containment,네트워크운영/보안관제,2026-08-01


In [10]:
# ============================================================
# 10. 임베딩 모델 로드
# ============================================================

embedding_model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

kb_embeddings = embedding_model.encode(
    kb_df["search_text"].tolist(),
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("문서 임베딩 크기:", kb_embeddings.shape)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

문서 임베딩 크기: (14, 384)


In [11]:
# ============================================================
# 11. 이상 이벤트 검색 문장 및 상세 RAG 검색
# ============================================================

event_features = [
    "protocol_type",
    "service",
    "flag",
    "duration",
    "src_bytes",
    "dst_bytes",
    "num_failed_logins",
    "logged_in",
    "count",
    "srv_count",
    "serror_rate",
    "srv_serror_rate",
    "rerror_rate",
    "srv_rerror_rate",
    "same_srv_rate",
    "diff_srv_rate",
    "dst_host_count",
    "dst_host_srv_count",
    "label",
    "binary_label",
    "anomaly_score"
]


def build_query(row):
    lines = [
        "NSL-KDD Isolation Forest 이상탐지 이벤트"
    ]

    for feature in event_features:
        if feature in row.index:
            lines.append(
                f"{feature}: {row.get(feature)}"
            )

    return "\n".join(lines)


def search_documents(query, top_k=5):
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0]

    similarities = kb_embeddings @ query_embedding
    indices = np.argsort(similarities)[::-1][:top_k]

    searched = kb_df.iloc[indices].copy()
    searched["similarity"] = similarities[indices]

    return searched.reset_index(drop=True)


sample_search = search_documents(
    build_query(top5_df.iloc[0]),
    top_k=5
)

display(
    sample_search[
        [
            "doc_id",
            "doc_type",
            "title",
            "attack_category",
            "similarity"
        ]
    ]
)

,doc_id,doc_type,title,attack_category,similarity
0,NET-IR-001,침해사고 사례,대량 SYN 연결을 이용한 서비스 거부 사고,DoS / SYN Flood,0.598099
1,NET-IR-004,침해사고 사례,FTP 계정 무차별 대입 및 로그인 성공,R2L / Brute Force,0.593292
2,NET-IR-005,침해사고 사례,취약 서비스 악용 후 원격 셸 획득,R2L / Exploitation,0.508941
3,NET-IR-003,침해사고 사례,다수 포트 스캔 후 원격서비스 접근 시도,Probe / Port Scan,0.482454
4,NET-IR-008,침해사고 사례,비정상 대용량 외부 전송 의심 사고,Data Exfiltration,0.477824


In [12]:
# ============================================================
# 12. 경량 Qwen 모델 로드
# ============================================================
# 기존 1.5B 모델보다 작은 0.5B 모델을 사용하여 생성 시간을 줄입니다.
# ============================================================

qwen_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(qwen_name)

if DEVICE == "cuda":
    qwen_model = AutoModelForCausalLM.from_pretrained(
        qwen_name,
        torch_dtype=torch.float16,
        device_map="auto",
        low_cpu_mem_usage=True
    )
else:
    qwen_model = AutoModelForCausalLM.from_pretrained(
        qwen_name,
        torch_dtype=torch.float32,
        low_cpu_mem_usage=True
    ).to("cpu")

qwen_model.eval()
print("Qwen 로드 완료:", qwen_name)
print("모델 실행 장치:", next(qwen_model.parameters()).device)


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen 로드 완료: Qwen/Qwen2.5-0.5B-Instruct
모델 실행 장치: cpu


In [13]:
# ============================================================
# 13. 빠른 Qwen 생성 함수
# ============================================================

def generate_qwen(system_text, user_text, max_new_tokens=420):
    messages = [
        {"role": "system", "content": system_text},
        {"role": "user", "content": user_text}
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2600
    )

    model_device = next(qwen_model.parameters()).device
    inputs = {
        key: value.to(model_device)
        for key, value in inputs.items()
    }

    with torch.inference_mode():
        output = qwen_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = output[0, inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()


In [ ]:
# ============================================================
# 14. 빠른 RAG 검색 및 Qwen 침해사고 보고서 생성
# ============================================================

report_rows = []

for index, row in top5_df.iterrows():
    print(f"[{index + 1}/{len(top5_df)}] {row['event_id']} 분석 중")

    query = build_query(row)
    searched = search_documents(query, top_k=3)

    context_parts = []
    document_details = []

    for rank, (_, doc) in enumerate(searched.iterrows(), start=1):
        context_parts.append(
            f"[검색 문서 {rank}위]\n"
            f"문서명: {doc['title']}\n"
            f"공격 유형: {doc['attack_category']}\n"
            f"개요: {doc['summary']}\n"
            f"탐지 근거: {doc['detection_points']}\n"
            f"대응절차: {doc['response_steps']}\n"
            f"확보 증적: {doc['evidence_to_collect']}\n"
            f"격상 기준: {doc['escalation_criteria']}\n"
            f"유사도: {doc['similarity']:.4f}"
        )

        document_details.append({
            "rank": rank,
            "doc_id": doc["doc_id"],
            "doc_type": doc["doc_type"],
            "title": doc["title"],
            "attack_category": doc["attack_category"],
            "summary": doc["summary"],
            "detection_points": doc["detection_points"],
            "response_steps": doc["response_steps"],
            "evidence_to_collect": doc["evidence_to_collect"],
            "escalation_criteria": doc["escalation_criteria"],
            "lessons_learned": doc["lessons_learned"],
            "owner": doc["owner"],
            "updated_at": doc["updated_at"],
            "similarity": round(float(doc["similarity"]), 4)
        })

    context = "\n\n".join(context_parts)

    prompt = f"""
[Isolation Forest 이상탐지 결과]
{query}

[RAG 검색 결과]
{context}

아래 7개 항목으로 간결한 침해사고 분석 보고서를 작성하라.
각 항목은 2~3문장으로 작성한다.

1. 사건 개요
2. 주요 이상 특징
3. 예상 공격 유형과 근거
4. RAG 검색 근거
5. 초동조치
6. 확보할 증적과 보고 기준
7. 최종 판단

주의사항:
- 비지도 이상탐지 결과만으로 공격을 확정하지 않는다.
- 확인 사실, 분석상 추정, 미확인 사항을 구분한다.
- 입력값과 검색 문서에 없는 사실은 만들지 않는다.
"""

    report = generate_qwen(
        system_text=(
            "너는 사이버보안센터의 네트워크 침해사고 분석관이다. "
            "제공된 탐지값과 지식문서만 근거로 간결하게 작성한다."
        ),
        user_text=prompt,
        max_new_tokens=420
    )

    output_row = row.to_dict()
    output_row.update({
        "top_document_title": searched.iloc[0]["title"],
        "retrieval_similarity": round(float(searched.iloc[0]["similarity"]), 4),
        "retrieved_documents": " | ".join(searched["title"].tolist()),
        "retrieved_document_details": json.dumps(document_details, ensure_ascii=False, indent=2),
        "qwen_report": report
    })

    report_rows.append(output_row)
    print(f"[{index + 1}/{len(top5_df)}] {row['event_id']} 완료")

report_df = pd.DataFrame(report_rows)
report_df.to_csv("exercise3_nsl_kdd_rag_reports.csv", index=False, encoding="utf-8-sig")

print("5건 보고서 생성 완료")
display(report_df[[
    "event_id", "label", "anomaly_score", "top_document_title",
    "retrieval_similarity", "retrieved_documents"
]])


[1/5] NSL-ANOM-001 분석 중
[1/5] NSL-ANOM-001 완료
[2/5] NSL-ANOM-002 분석 중
[2/5] NSL-ANOM-002 완료
[3/5] NSL-ANOM-003 분석 중
[3/5] NSL-ANOM-003 완료
[4/5] NSL-ANOM-004 분석 중


In [ ]:
# ============================================================
# 15. HTML 대시보드 생성
# ============================================================
# HTML은 Qwen으로 생성하지 않고 파이썬으로 즉시 만듭니다.
# ============================================================

def esc(value):
    if value is None:
        return "-"
    try:
        if pd.isna(value):
            return "-"
    except Exception:
        pass
    return html.escape(str(value))


def report_to_html(text):
    return esc(text).replace("\n", "<br>")

cards = []

for _, row in report_df.iterrows():
    try:
        docs = json.loads(row.get("retrieved_document_details", "[]"))
    except Exception:
        docs = []

    doc_html = ""
    for doc in docs:
        doc_html += f"""
        <details class="doc-detail">
            <summary>{esc(doc.get('rank'))}위 · {esc(doc.get('title'))}</summary>
            <p><b>공격 유형:</b> {esc(doc.get('attack_category'))}</p>
            <p><b>유사도:</b> {esc(doc.get('similarity'))}</p>
            <p><b>개요:</b> {esc(doc.get('summary'))}</p>
            <p><b>탐지 근거:</b> {esc(doc.get('detection_points'))}</p>
            <p><b>대응절차:</b> {esc(doc.get('response_steps'))}</p>
            <p><b>확보 증적:</b> {esc(doc.get('evidence_to_collect'))}</p>
            <p><b>격상 기준:</b> {esc(doc.get('escalation_criteria'))}</p>
        </details>
        """

    score = float(row.get("anomaly_score", 0))
    cards.append(f"""
    <section class="card" data-text="{esc(row.get('event_id'))} {esc(row.get('label'))} {esc(row.get('top_document_title'))}">
        <div class="card-head">
            <div>
                <h2>{esc(row.get('event_id'))}</h2>
                <span class="badge">{esc(row.get('model_prediction'))}</span>
            </div>
            <div class="score">이상점수 {score:.4f}</div>
        </div>
        <div class="grid">
            <p><b>프로토콜</b><br>{esc(row.get('protocol_type'))}</p>
            <p><b>서비스</b><br>{esc(row.get('service'))}</p>
            <p><b>상태</b><br>{esc(row.get('flag'))}</p>
            <p><b>원본 라벨</b><br>{esc(row.get('label'))}</p>
            <p><b>src_bytes</b><br>{esc(row.get('src_bytes'))}</p>
            <p><b>dst_bytes</b><br>{esc(row.get('dst_bytes'))}</p>
            <p><b>count</b><br>{esc(row.get('count'))}</p>
            <p><b>srv_count</b><br>{esc(row.get('srv_count'))}</p>
        </div>
        <h3>RAG 검색 결과</h3>
        <p><b>최상위 문서:</b> {esc(row.get('top_document_title'))}</p>
        <p><b>최상위 유사도:</b> {esc(row.get('retrieval_similarity'))}</p>
        {doc_html}
        <details class="report-detail" open>
            <summary>Qwen 침해사고 분석 보고서</summary>
            <div class="report">{report_to_html(row.get('qwen_report', ''))}</div>
        </details>
    </section>
    """)

cards_html = "\n".join(cards)

generated_html = f"""<!DOCTYPE html>
<html lang="ko">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>NSL-KDD 이상탐지 RAG 대시보드</title>
<style>
* {{ box-sizing: border-box; }}
body {{ margin: 0; background: #0b1220; color: #e5edf8; font-family: Arial, sans-serif; }}
header {{ position: sticky; top: 0; z-index: 5; padding: 22px; background: #111b2e; border-bottom: 1px solid #273755; }}
header h1 {{ margin: 0 0 12px; font-size: 24px; }}
input {{ width: 100%; max-width: 520px; padding: 12px; border-radius: 8px; border: 1px solid #405276; background: #0b1220; color: white; }}
main {{ max-width: 1200px; margin: auto; padding: 24px; }}
.card {{ background: #111b2e; border: 1px solid #273755; border-radius: 14px; padding: 22px; margin-bottom: 22px; }}
.card-head {{ display: flex; justify-content: space-between; gap: 15px; align-items: center; }}
h2, h3 {{ margin-top: 0; }}
h3 {{ margin-top: 24px; color: #9ec5ff; }}
.badge {{ display: inline-block; padding: 5px 9px; border-radius: 999px; background: #7f1d1d; color: #fecaca; font-size: 13px; }}
.score {{ font-weight: bold; color: #fbbf24; }}
.grid {{ display: grid; grid-template-columns: repeat(4, 1fr); gap: 10px; margin: 18px 0; }}
.grid p {{ margin: 0; padding: 12px; background: #0b1220; border-radius: 9px; line-height: 1.5; }}
details {{ margin: 10px 0; background: #0b1220; border: 1px solid #273755; border-radius: 8px; padding: 11px; }}
summary {{ cursor: pointer; font-weight: bold; color: #bfdbfe; }}
.report {{ margin-top: 14px; line-height: 1.75; }}
@media (max-width: 760px) {{ .grid {{ grid-template-columns: repeat(2, 1fr); }} .card-head {{ align-items: flex-start; flex-direction: column; }} }}
</style>
</head>
<body>
<header>
    <h1>NSL-KDD 이상탐지 · RAG 침해사고 분석</h1>
    <input id="search" placeholder="이벤트, 공격명, 문서명 검색">
</header>
<main id="cards">{cards_html}</main>
<script>
const input = document.getElementById('search');
input.addEventListener('input', () => {{
    const keyword = input.value.toLowerCase();
    document.querySelectorAll('.card').forEach(card => {{
        card.style.display = card.dataset.text.toLowerCase().includes(keyword) ? '' : 'none';
    }});
}});
</script>
</body>
</html>"""

with open("exercise3_qwen_anomaly_dashboard.html", "w", encoding="utf-8") as file:
    file.write(generated_html)

print("HTML 대시보드 저장 완료")


In [ ]:
# ============================================================
# 16. HTML 미리보기 및 결과 파일 다운로드
# ============================================================

display(HTML(generated_html))

files.download("exercise3_nsl_kdd_top5_anomalies.csv")
files.download("exercise3_nsl_kdd_rag_reports.csv")
files.download("exercise3_qwen_anomaly_dashboard.html")

if __import__("os").path.exists("center_network_incident_rag_knowledge.csv"):
    files.download("center_network_incident_rag_knowledge.csv")
